# Experiment 06 — Direct per-series Ridge (a different paradigm)

A deliberate architectural break from notebooks 03–05. Those use **one global LightGBM, forecasting recursively** (each predicted day is fed back into the lag features of the next). This notebook tests the opposite design — close to what the most-forked public notebooks (~0.3735) appear to do:

| | Notebooks 03–05 | This notebook (06) |
|---|---|---|
| Learner | LightGBM (trees) | **Ridge (linear)** |
| Forecast | recursive, day-by-day | **direct**: all 16 days at once |
| Feedback | own predictions feed the lags | **none** — no recursion, no error accumulation |
| Granularity | one model for everything | **one model per store×family** (1,782) |
| Trend | trees can't extrapolate | **linear, extrapolates** |
| Main signal | sales lags | **`onpromotion` + calendar** (all known for the test horizon) |

Each of the 1,782 series gets its own tiny regression of `log1p(sales)` on features that are **fully known at forecast time** — calendar, Fourier seasonality, holiday/geo flags and the (known-future) promotion count. No sales lags at all: the intercept carries the level, the trend term carries the drift. All 16 test days are predicted in one shot.

**Why this might close the gap.** GBM with recursive lags accumulates error over 16 days and cannot extrapolate trend; a per-series linear model does both, and uses the known future `onpromotion` directly. The real hope is the **blend** of this with our best GBM (`geo_blend25`, public 0.38899): two maximally different paradigms whose errors should cancel.

Validation is unchanged: the same 3 rolling windows and the same gate (beat the baseline on **all 3**). The engine, however, is far simpler — no iterative loop — and runs in a couple of minutes.

> 🇷🇺 Сознательный слом архитектуры: вместо одного глобального LightGBM с рекурсивным прогнозом — 1782 крошечные линейные регрессии Ridge, по одной на ряд store×family, прямой прогноз всех 16 дней без рекурсии. Признаки только известные на момент прогноза: календарь, Fourier, праздники/гео и (известный для теста) `onpromotion`; лагов продаж нет — уровень несёт свободный член, дрейф — тренд. Та же валидация на 3 окнах и тот же гейт. Главная надежда — бленд с нашим лучшим GBM (`geo_blend25`, паблик 0.38899): две максимально разные парадигмы.

In [1]:
import time

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

In [2]:
_t0 = time.time()

ART_DIR   = "../artifacts/exp06"
EXP05_ART = "../artifacts/exp05"   # cached geo_blend25 window predictions (our best GBM)
os.makedirs(ART_DIR, exist_ok=True)

# Knobs / Ручки
TRAIN_START = pd.Timestamp("2015-01-01")  # history per series (>=2 yearly cycles for Fourier)
ALPHA       = 10.0                        # Ridge regularization (features are ~standardized)
MIN_NONZERO = 30                          # below this many nonzero days -> recent-mean fallback
CAP_MULT    = 1.5                         # cap a series' prediction at CAP_MULT x its historical max (linear models explode on erratic series)

WINDOWS = [
    ("W1", pd.Timestamp("2017-06-15"), pd.Timestamp("2017-06-30")),
    ("W2", pd.Timestamp("2017-07-15"), pd.Timestamp("2017-07-30")),
    ("W3", pd.Timestamp("2017-07-31"), pd.Timestamp("2017-08-15")),
]
WNAMES = [w for w, _, _ in WINDOWS]
TEST_START, TEST_END = pd.Timestamp("2017-08-16"), pd.Timestamp("2017-08-31")

print(f"TRAIN_START={TRAIN_START.date()}, ALPHA={ALPHA}, MIN_NONZERO={MIN_NONZERO}")

TRAIN_START=2015-01-01, ALPHA=10.0, MIN_NONZERO=30


In [3]:
# Load the feature superset; reshape into (dates x series) matrices (data is rectangular)
# Загружаем супермножество фич; укладываем в матрицы (даты x ряды) — данные прямоугольные
DF = (
    pd.read_parquet("../artifacts/features.parquet")
    .sort_values(["date", "store_nbr", "family"])
    .reset_index(drop=True)
)
DATES = np.sort(DF["date"].unique())
N_DATES = len(DATES)
N_SERIES = DF.groupby("date").size().iloc[0]
assert len(DF) == N_DATES * N_SERIES, "calendar is not rectangular"
DATE_IDX = {d: i for i, d in enumerate(DATES)}
ref = DF.iloc[:N_SERIES][["store_nbr", "family"]].reset_index(drop=True)
SALES_TRUE = DF["sales"].to_numpy(dtype=float).reshape(N_DATES, N_SERIES)  # test rows are NaN
DT = pd.to_datetime(DATES)

# Design matrix: ONLY features known at forecast time (no sales lags)
# Дизайн-матрица: ТОЛЬКО известные на момент прогноза признаки (без лагов продаж)
doy = DF["day_of_year"].to_numpy(dtype=float)
cont = np.column_stack([
    DF["date_index"].to_numpy(dtype=float),        # linear trend
    np.log1p(DF["onpromotion"].to_numpy(dtype=float)),  # known-future promo signal
])
# Standardize continuous cols on train rows only (avoid using the test distribution)
# Стандартизуем непрерывные колонки только по train-строкам (распределение теста не используем)
train_rows = DF["date"].to_numpy() < np.datetime64(TEST_START)
mu = cont[train_rows].mean(axis=0)
sd = cont[train_rows].std(axis=0) + 1e-9
cont = (cont - mu) / sd

fourier = np.column_stack([
    f(2 * np.pi * k * doy / 365.0) for k in (1, 2, 3) for f in (np.sin, np.cos)
])  # 3 yearly harmonics
dow = pd.get_dummies(DF["day_of_week"], prefix="dow", drop_first=True).to_numpy(dtype=float)  # 6 weekday dummies

FLAGS = ["is_holiday_national", "is_holiday_local", "is_holiday_regional", "day_before_holiday",
         "is_black_friday", "is_cyber_monday", "is_terremoto", "is_navidad", "is_dia_madre",
         "is_futbol", "is_dia_trabajo", "is_primer_dia", "is_dia_difuntos", "work_day", "is_payday"]
flags = DF[FLAGS].to_numpy(dtype=float)

# Safe sales lags: >=16 days back are known for the entire 16-day horizon (no recursion); yearly too.
# log1p-transformed, NaN (early history) -> 0. This is the proper "direct" design.
# Безопасные лаги продаж: >=16 дней и годовые известны на весь горизонт без рекурсии.
# log1p, NaN (ранняя история) -> 0. Это правильный «прямой» дизайн.
SAFE_LAGS = ["lag_21", "lag_28", "lag_42", "lag_56", "lag_364", "lag_365"]
lags = np.log1p(np.nan_to_num(DF[SAFE_LAGS].to_numpy(dtype=float), nan=0.0))
mu_l = lags[train_rows].mean(axis=0); sd_l = lags[train_rows].std(axis=0) + 1e-9
lags = (lags - mu_l) / sd_l

X = np.hstack([cont, fourier, dow, flags, lags]).astype(np.float32)
N_FEAT = X.shape[1]
XR = X.reshape(N_DATES, N_SERIES, N_FEAT)   # XR[date, series, feature]
YL = np.log1p(SALES_TRUE)                    # target on log scale (NaN on test rows)
print(f"{N_DATES} dates x {N_SERIES} series; design matrix {N_FEAT} features "
      f"(2 cont + {fourier.shape[1]} fourier + {dow.shape[1]} dow + {len(FLAGS)} flags + {len(SAFE_LAGS)} safe-lags)")

1704 dates x 1782 series; design matrix 35 features (2 cont + 6 fourier + 6 dow + 15 flags + 6 safe-lags)


In [4]:
# Direct Ridge engine: one model per series, predict the whole window at once
# Прямой Ridge-движок: одна модель на ряд, прогноз всего окна сразу
def rmsle_mat(P, Y):
    return float(np.sqrt(np.mean((np.log1p(np.clip(P, 0, None)) - np.log1p(Y)) ** 2)))


def zero_rule_mask(first_idx):
    return np.nansum(SALES_TRUE[first_idx - 21:first_idx], axis=0) == 0


def ridge_direct(window_first_idx, win_idxs):
    """Per-series direct forecast for the 16 window days (no recursion).
    Series with too few nonzero days fall back to their recent mean."""
    date_ord = np.arange(N_DATES)
    train_time = np.asarray(DT >= TRAIN_START) & (date_ord < window_first_idx)
    P = np.empty((len(win_idxs), N_SERIES))
    n_fallback = 0
    for s in range(N_SERIES):
        ys = YL[:, s]
        m = train_time & ~np.isnan(ys)
        nz = int(np.nansum(SALES_TRUE[m, s] > 0))
        if nz < MIN_NONZERO:                       # intermittent / too short -> recent mean
            rec = np.nanmean(SALES_TRUE[window_first_idx - 21:window_first_idx, s])
            P[:, s] = 0.0 if np.isnan(rec) else max(rec, 0.0)
            n_fallback += 1
            continue
        model = Ridge(alpha=ALPHA)
        model.fit(XR[m, s, :], ys[m])
        pred = np.expm1(model.predict(XR[win_idxs, s, :])).clip(0)
        hist_max = np.nanmax(SALES_TRUE[m, s])              # guard: linear fit can explode on erratic series
        if np.isfinite(hist_max) and hist_max > 0:
            pred = np.minimum(pred, CAP_MULT * hist_max)
        P[:, s] = pred
    return P, n_fallback


# Sanity: design matrix has no NaN/inf, promo column varies / Проверка: в матрице нет NaN/inf, промо варьируется
assert np.isfinite(X).all(), "design matrix has NaN/inf"
print("OK: design matrix is finite; ready to fit", N_SERIES, "per-series models")

OK: design matrix is finite; ready to fit 1782 per-series models


In [5]:
# Main runs: ridge_direct alone and blended with our best GBM (geo_blend25)
# Blend is a log-space average (consistent with RMSLE); geo_blend25 window preds come from exp05 cache.
# Основные прогоны: ridge_direct сам по себе и в бленде с лучшим GBM (geo_blend25)
# Бленд — среднее в log-пространстве (согласовано с RMSLE); прогнозы geo_blend25 из кэша exp05.
RESULTS = {}
rows = []

for wname, ws, we in WINDOWS:
    print(f"\n=== {wname}: {ws.date()} .. {we.date()} ===")
    win_idxs = [i for i in range(N_DATES) if ws <= DT[i] <= we]
    assert len(win_idxs) == 16
    si = win_idxs[0]
    Y = SALES_TRUE[win_idxs]
    zmask = zero_rule_mask(si)

    t = time.time()
    cache = os.path.join(ART_DIR, f"pred_{wname}__ridge_sf.npy")
    if os.path.exists(cache):
        P_ridge = np.load(cache)
    else:
        P_ridge, nfb = ridge_direct(si, win_idxs)
        P_ridge[:, zmask] = 0.0
        np.save(cache, P_ridge)
        print(f"  fitted in {time.time() - t:.0f}s, {nfb} series on recent-mean fallback")
    RESULTS[(wname, "ridge_sf")] = P_ridge

    # Our best GBM on this window (from exp05) and the two-paradigm blend
    # Наш лучший GBM на этом окне (из exp05) и бленд двух парадигм
    P_gbm = np.load(os.path.join(EXP05_ART, f"pred_{wname}__geo_blend25.npy"))
    P_blend = np.expm1(0.5 * np.log1p(P_ridge.clip(0)) + 0.5 * np.log1p(P_gbm.clip(0)))
    RESULTS[(wname, "geo_blend25")] = P_gbm
    RESULTS[(wname, "ridge_gbm_blend")] = P_blend

    for scheme in ["ridge_sf", "geo_blend25", "ridge_gbm_blend"]:
        sc = rmsle_mat(RESULTS[(wname, scheme)], Y)
        rows.append({"window": wname, "scheme": scheme, "rmsle": sc})
        print(f"  {scheme:18s} RMSLE={sc:.5f}")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(ART_DIR, "results.csv"), index=False)
print(f"\nElapsed: {(time.time() - _t0) / 60:.1f} min")


=== W1: 2017-06-15 .. 2017-06-30 ===
  ridge_sf           RMSLE=0.60147
  geo_blend25        RMSLE=0.38173
  ridge_gbm_blend    RMSLE=0.44103

=== W2: 2017-07-15 .. 2017-07-30 ===
  ridge_sf           RMSLE=0.59419
  geo_blend25        RMSLE=0.38652
  ridge_gbm_blend    RMSLE=0.44421

=== W3: 2017-07-31 .. 2017-08-15 ===
  ridge_sf           RMSLE=0.58294
  geo_blend25        RMSLE=0.40187
  ridge_gbm_blend    RMSLE=0.44936

Elapsed: 0.1 min


In [6]:
# Summary and gate
# geo_blend25 is the baseline (our best GBM, public 0.38899). ridge_sf and the blend each
# must beat it on ALL 3 windows to be deployed.
# Сводка и гейт
# Бейзлайн — geo_blend25 (наш лучший GBM, паблик 0.38899). И ridge_sf, и бленд обязаны
# побить его на ВСЕХ 3 окнах, чтобы пойти в сабмит.
piv = res.pivot(index="scheme", columns="window", values="rmsle")
piv["mean"] = piv[WNAMES].mean(axis=1)
piv = piv.sort_values("mean")
display(piv.round(5))

def beats(cand, base="geo_blend25"):
    return all(piv.loc[cand, w] < piv.loc[base, w] for w in WNAMES)

print("\nvs geo_blend25 (must win all 3 windows):")
for c in ["ridge_sf", "ridge_gbm_blend"]:
    flags = "  ".join(f"{w}:{piv.loc[c, w] - piv.loc['geo_blend25', w]:+.5f}" for w in WNAMES)
    print(f"  {c:18s} {flags}   gate={'PASS' if beats(c) else 'fail'}")

GATED = [c for c in ["ridge_sf", "ridge_gbm_blend"] if beats(c)]
print("\nGate passed by:", GATED if GATED else "NOBODY")

window,W1,W2,W3,mean
scheme,,,,
geo_blend25,0.38173,0.38652,0.40187,0.39004
ridge_gbm_blend,0.44103,0.44421,0.44936,0.44486
ridge_sf,0.60147,0.59419,0.58294,0.59287



vs geo_blend25 (must win all 3 windows):
  ridge_sf           W1:+0.21974  W2:+0.20766  W3:+0.18107   gate=fail
  ridge_gbm_blend    W1:+0.05930  W2:+0.05768  W3:+0.04748   gate=fail

Gate passed by: NOBODY


In [7]:
# Final: refit on full train, predict the test, build submission files
# Always emit the pure ridge submission (diagnostic) and the ridge+GBM blend (the real candidate).
# Финал: переобучение на всём train, прогноз теста, сборка файлов сабмишна
# Всегда сохраняем чистый ridge (диагностика) и бленд ridge+GBM (главный кандидат).
test_idxs = [i for i in range(N_DATES) if TEST_START <= DT[i] <= TEST_END]
assert len(test_idxs) == 16
si_test = test_idxs[0]
test = pd.read_csv("../data/test.csv", parse_dates=["date"])


def make_submission(P_test, name):
    long = pd.concat(
        [pd.DataFrame({"date": DATES[i], "store_nbr": ref["store_nbr"].values,
                       "family": ref["family"].values, "sales": P_test[j]})
         for j, i in enumerate(test_idxs)],
        ignore_index=True,
    )
    sub = test.merge(long, on=["date", "store_nbr", "family"], how="left")[["id", "sales"]]
    assert sub["sales"].notna().all()
    sub.to_csv(f"../submission_{name}.csv", index=False)
    print(f"Saved: submission_{name}.csv  (total {sub['sales'].sum():,.0f})")


cache = os.path.join(ART_DIR, "pred_TEST__ridge_sf.npy")
if os.path.exists(cache):
    P_ridge_test = np.load(cache)
else:
    t = time.time()
    P_ridge_test, nfb = ridge_direct(si_test, test_idxs)
    P_ridge_test[:, zero_rule_mask(si_test)] = 0.0
    np.save(cache, P_ridge_test)
    print(f"test fitted in {time.time() - t:.0f}s, {nfb} fallback series")

make_submission(P_ridge_test, "06_ridge")

# Blend with our deployed best GBM submission (public 0.38899), in log space
# Бленд с задеплоенным лучшим GBM-сабмишном (паблик 0.38899), в log-пространстве
gbm_sub = pd.read_csv("../submission_05_geo_blend25.csv").sort_values("id").reset_index(drop=True)
ridge_sub = pd.read_csv("../submission_06_ridge.csv").sort_values("id").reset_index(drop=True)
assert (gbm_sub["id"] == ridge_sub["id"]).all()
blend = gbm_sub[["id"]].copy()
blend["sales"] = np.expm1(0.5 * np.log1p(ridge_sub["sales"]) + 0.5 * np.log1p(gbm_sub["sales"]))
blend.to_csv("../submission_06_ridge_gbm_blend.csv", index=False)
print(f"Saved: submission_06_ridge_gbm_blend.csv  (total {blend['sales'].sum():,.0f})")
print(f"\nTotal runtime: {(time.time() - _t0) / 60:.1f} min")

Saved: submission_06_ridge.csv  (total 15,506,326)
Saved: submission_06_ridge_gbm_blend.csv  (total 13,215,209)

Total runtime: 0.1 min


## Conclusions

**Negative result: per-series direct Ridge is much weaker than the pooled global GBM, and the blend does not help.** The hypothesis "the public-notebook wall (~0.3735) is per-series linear regression" is rejected.

Mean RMSLE over the 3 windows:

| Variant | Ridge alone | Ridge+GBM blend | GBM baseline (`geo_blend25`) |
|---|---:|---:|---:|
| v1 — no sales history, prediction capped at 1.5× series max | 0.751 | 0.501 | **0.390** |
| v2 — + safe lags (≥16d and yearly) | 0.593 | 0.445 | **0.390** |

Each addition narrows the gap, but Ridge stays far behind, and blending a 0.59 model with a 0.39 model only drags the result to 0.44. No variant passes the gate.

**Why linear loses:**
1. **No pooling across series.** The global GBM learns from all 1,782 series at once, so noisy low-volume series borrow strength from similar ones. 1,782 independent Ridge models each fit in isolation on limited, noisy data.
2. **Trees capture interactions** (promo×family, holiday×store-type) that a linear model misses.
3. **Short recursive lags are stronger.** The GBM uses `lag_1`/`lag_7`; a direct model is limited to lags ≥16, which predict less well.

**Takeaway.** The path forward is not linear models but stronger trees. The diagnostic `submission_06_*` files were removed — at RMSLE 0.44–0.59 they would only waste leaderboard slots.

## Выводы

**Отрицательный результат: per-series линейная direct-модель намного слабее пулингового глобального GBM, и бленд не помогает.** Гипотеза «стена публичных ноутбуков (~0.3735) — это линейная регрессия на каждый ряд» отвергнута.

Среднее RMSLE по 3 окнам:

| Вариант | Ridge соло | Бленд Ridge+GBM | Бейзлайн GBM (`geo_blend25`) |
|---|---:|---:|---:|
| v1 — без истории продаж, прогноз обрезан на 1.5× максимума ряда | 0.751 | 0.501 | **0.390** |
| v2 — + безопасные лаги (≥16 дней и годовые) | 0.593 | 0.445 | **0.390** |

Каждое добавление сокращает разрыв, но Ridge остаётся далеко позади, а смешивание модели 0.59 с моделью 0.39 лишь тянет результат к 0.44. Ни один вариант не проходит гейт.

**Почему линейная проигрывает:**
1. **Нет объединения рядов.** Глобальный GBM учится сразу на всех 1782 рядах, поэтому шумные мелкие ряды заимствуют силу у похожих. 1782 независимые Ridge варятся каждая в своих скудных и шумных данных.
2. **Деревья ловят взаимодействия** (промо×категория, праздник×тип магазина), которые линейная модель упускает.
3. **Короткие рекурсивные лаги сильнее.** GBM использует `lag_1`/`lag_7`; direct-модель ограничена лагами ≥16, которые предсказывают хуже.

**Вывод.** Путь вперёд — не линейные модели, а более сильные деревья. Диагностические файлы `submission_06_*` удалены — при RMSLE 0.44–0.59 они лишь потратили бы слоты на лидерборде.